# Synthetic Patient Generation

## Research project

**Objective:** generate synthetic patient phenotype profiles from disease-associated HPO annotations while accounting for the reported frequency of each phenotype.

This notebook covers:
1. Loading and inspecting HPO annotations.
2. Converting phenotype frequency annotations into probabilities.
3. Aggregating duplicate disease–phenotype annotations.
4. Building disease-level frequency profiles.
5. Sampling phenotype profiles and validating the synthetic-patient generator.

Reusable implementation is kept in the `src/` package. This notebook is used for exploration, experiments, and validation.


## 1. Project setup and data loading

We first locate the project root, load the HPO ontology and annotation file, and retain positive phenotypic annotations (`aspect == "P"` with no exclusion qualifier).

The project root is detected from the current working directory and its parent directories so the notebook can be run whether the working directory is the repository root or the `notebooks/` folder.


In [1]:
from pathlib import Path
import sys
import random
import importlib
from collections import Counter

import pandas as pd
import obonet
import networkx as nx

# Find the repository root from the current working directory or its parents.
candidate_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (
        path for path in candidate_roots
        if (path / "data" / "hpo").exists()
        and (path / "src").exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. "
        "Run this notebook from the repository root or its notebooks/ folder."
    )

HPO_DIR = PROJECT_ROOT / "data" / "hpo"
OBO_FILE = HPO_DIR / "hp-base.obo"
HPOA_FILE = HPO_DIR / "phenotype.hpoa"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Ontology file exists:", OBO_FILE.exists())
print("Annotation file exists:", HPOA_FILE.exists())

if not OBO_FILE.exists() or not HPOA_FILE.exists():
    raise FileNotFoundError("One or both HPO input files are missing.")


Project root: D:\uni_ 3\ProiectCercetare\hpo-similarity-synthetic-vs-clinical
Ontology file exists: True
Annotation file exists: True


In [3]:
# Load the HPO ontology.
hpo_graph = obonet.read_obo(OBO_FILE)

# Load disease–phenotype annotations.
annotations = pd.read_csv(
    HPOA_FILE,
    sep="\t",
    comment="#",
    dtype=str
)

# Keep positive phenotypic annotations.
phenotype_annotations = annotations[
    (annotations["aspect"] == "P") &
    (annotations["qualifier"].isna())
].copy()

print("Ontology nodes:", hpo_graph.number_of_nodes())
print("Ontology edges:", hpo_graph.number_of_edges())
print("Positive phenotype annotations:", len(phenotype_annotations))


Ontology nodes: 19904
Ontology edges: 24453
Positive phenotype annotations: 268179


## 2. Explore the frequency annotations

The HPO annotation file contains phenotype frequency values in several formats, including fractions and HPO frequency categories. We inspect the source values before converting them into numeric probabilities.

We also inspect duplicate disease–phenotype pairs because the same pair can have multiple annotations.


In [4]:
print("Available columns:")
print(phenotype_annotations.columns.tolist())

print("\nFrequency examples:")
print(phenotype_annotations["frequency"].value_counts(dropna=False).head(15))

print("\nSample annotations:")
display(
    phenotype_annotations[
        ["database_id", "hpo_id", "frequency"]
    ].head(15)
)

Available columns:
['database_id', 'disease_name', 'qualifier', 'hpo_id', 'reference', 'evidence', 'onset', 'frequency', 'sex', 'modifier', 'aspect', 'biocuration']

Frequency examples:
frequency
NaN           51902
HP:0040283    45420
HP:0040282    40398
HP:0040281    25753
1/1           18964
2/2            7337
HP:0040284     7026
1/2            5901
3/3            4016
1/3            3966
1/4            3205
4/4            2748
1/5            2212
2/3            1860
1/6            1722
Name: count, dtype: int64

Sample annotations:


,database_id,hpo_id,frequency
0,OMIM:619340,HP:0011097,1/2
1,OMIM:619340,HP:0002187,1/1
2,OMIM:619340,HP:0001518,1/2
3,OMIM:619340,HP:0032792,1/2
4,OMIM:619340,HP:0011451,1/2
5,OMIM:619340,HP:0010851,2/2
6,OMIM:619340,HP:0001789,1/2
7,OMIM:619340,HP:0200134,2/2
10,OMIM:619340,HP:0002643,2/2
11,OMIM:609153,HP:0002378,NaN


In [5]:
frequency_terms = [
    "HP:0040281",
    "HP:0040282",
    "HP:0040283",
    "HP:0040284"
]

for term in frequency_terms:
    print(term, "->", hpo_graph.nodes[term].get("name"))

HP:0040281 -> Very frequent
HP:0040282 -> Frequent
HP:0040283 -> Occasional
HP:0040284 -> Very rare


In [6]:
for term in frequency_terms:
    print(f"\n{term}")
    print(hpo_graph.nodes[term])


HP:0040281
{'name': 'Very frequent', 'def': '"Present in 80% to 99% of the cases." []', 'synonym': ['"Very frequent (99-80%)" EXACT []'], 'xref': ['ORPHA:453311'], 'is_a': ['HP:0040279'], 'property_value': ['dcterms:creator https://orcid.org/0000-0002-5316-1399', 'dcterms:date "2016-12-01T11:44:56Z" xsd:dateTime']}

HP:0040282
{'name': 'Frequent', 'def': '"Present in 30% to 79% of the cases." []', 'synonym': ['"Frequent (79-30%)" EXACT []'], 'xref': ['ORPHA:453312'], 'is_a': ['HP:0040279'], 'property_value': ['dcterms:creator https://orcid.org/0000-0002-5316-1399', 'dcterms:date "2016-12-01T11:44:56Z" xsd:dateTime']}

HP:0040283
{'name': 'Occasional', 'def': '"Present in 5% to 29% of the cases." []', 'synonym': ['"Occasional (29-5%)" EXACT []'], 'xref': ['ORPHA:453313'], 'is_a': ['HP:0040279'], 'property_value': ['dcterms:creator https://orcid.org/0000-0002-5316-1399', 'dcterms:date "2016-12-01T11:44:56Z" xsd:dateTime']}

HP:0040284
{'name': 'Very rare', 'def': '"Present in 1% to 4% o

In [7]:
# Check duplicate disease-phenotype pairs
duplicate_pairs = phenotype_annotations.duplicated(
    subset=["database_id", "hpo_id"],
    keep=False
)

print("Total annotations:", len(phenotype_annotations))
print("Annotations in duplicate pairs:", duplicate_pairs.sum())

display(
    phenotype_annotations.loc[
        duplicate_pairs,
        ["database_id", "hpo_id", "frequency"]
    ].head(20)
)

Total annotations: 268179
Annotations in duplicate pairs: 1046


,database_id,hpo_id,frequency
124,OMIM:612567,HP:0025084,2/2
125,OMIM:612567,HP:0025084,1/1
152,OMIM:614116,HP:0000726,18/18
153,OMIM:614116,HP:0000726,NaN
1233,OMIM:122600,HP:0002650,3/3
1234,OMIM:122600,HP:0002650,23/23
1789,OMIM:616271,HP:0001252,8/14
1790,OMIM:616271,HP:0001252,3/5
3216,OMIM:619817,HP:0007589,1/1
3217,OMIM:619817,HP:0007589,1/1


In [8]:
duplicate_details = phenotype_annotations.loc[
    duplicate_pairs,
    [
        "database_id",
        "hpo_id",
        "frequency",
        "reference",
        "evidence",
        "biocuration"
    ]
]

display(duplicate_details.head(30))

,database_id,hpo_id,frequency,reference,evidence,biocuration
124,OMIM:612567,HP:0025084,2/2,PMID:19890111,PCS,HPO:skoehler[2018-10-08];HPO:probinson[2020-11...
125,OMIM:612567,HP:0025084,1/1,PMID:21519361,PCS,HPO:probinson[2022-02-27]
152,OMIM:614116,HP:0000726,18/18,PMID:21532572,PCS,HPO:skoehler[2012-11-20]
153,OMIM:614116,HP:0000726,NaN,OMIM:614116,TAS,HPO:skoehler[2012-11-20]
1233,OMIM:122600,HP:0002650,3/3,PMID:23335591,PCS,HPO:probinson[2015-02-21];HPO:probinson[2022-0...
1234,OMIM:122600,HP:0002650,23/23,PMID:25564734,PCS,HPO:probinson[2022-08-03]
1789,OMIM:616271,HP:0001252,8/14,PMID:25597510,PCS,HPO:skoehler[2015-05-31];HPO:probinson[2022-06...
1790,OMIM:616271,HP:0001252,3/5,PMID:25597511,PCS,HPO:probinson[2022-06-04]
3216,OMIM:619817,HP:0007589,1/1,PMID:9185503,PCS,HPO:probinson[2022-05-12]
3217,OMIM:619817,HP:0007589,1/1,PMID:14675179,PCS,HPO:probinson[2022-05-12]


In [9]:
# Count annotations per disease-phenotype pair
pair_counts = (
    phenotype_annotations
    .groupby(["database_id", "hpo_id"])
    .size()
)

print("Total disease-phenotype pairs:", len(pair_counts))
print("Pairs with multiple annotations:", (pair_counts > 1).sum())
print("Maximum annotations for one pair:", pair_counts.max())

print("\nDistribution of annotations per pair:")
print(pair_counts.value_counts().sort_index())

Total disease-phenotype pairs: 267579
Pairs with multiple annotations: 446
Maximum annotations for one pair: 16

Distribution of annotations per pair:
1     267133
2        355
3         65
4         13
5          8
6          1
7          2
13         1
16         1
Name: count, dtype: int64


## 3. Convert and aggregate phenotype frequencies

The conversion and aggregation logic is implemented in `src/phenotype_frequency.py`.

- `frequency_to_probability()` converts a source frequency value into a numeric probability.
- `aggregate_phenotype_frequencies()` combines repeated annotations for the same disease–phenotype pair.
- Unknown frequencies remain missing (`None`/`NaN`) rather than being treated as zero.

For duplicate disease–phenotype pairs, the current implementation uses the mean of available numeric frequencies and records annotation counts.


In [10]:
import importlib
import src.phenotype_frequency

importlib.reload(src.phenotype_frequency)

from src.phenotype_frequency import (
    frequency_to_probability,
    aggregate_phenotype_frequencies
)

test_frequencies = [
    "1/2",
    "3/5",
    "25%",
    "80%",
    "100%",
    "HP:0040280",
    "HP:0040281",
    "HP:0040282",
    "HP:0040283",
    "HP:0040284",
    None
]

for value in test_frequencies:
    print(value, "->", frequency_to_probability(value))

1/2 -> 0.5
3/5 -> 0.6
25% -> 0.25
80% -> 0.8
100% -> 1.0
HP:0040280 -> 1.0
HP:0040281 -> 0.895
HP:0040282 -> 0.545
HP:0040283 -> 0.16999999999999998
HP:0040284 -> 0.025
None -> None


In [11]:

from src.phenotype_frequency import aggregate_phenotype_frequencies

aggregated_frequencies = aggregate_phenotype_frequencies(
    phenotype_annotations
)

print("Original annotations:", len(phenotype_annotations))
print("Aggregated disease-phenotype pairs:", len(aggregated_frequencies))

display(aggregated_frequencies.head())


Original annotations: 268179
Aggregated disease-phenotype pairs: 267579


,database_id,hpo_id,frequency,annotation_count,unknown_frequency_count
0,DECIPHER:1,HP:0000252,NaN,1,1
1,DECIPHER:1,HP:0001249,NaN,1,1
2,DECIPHER:1,HP:0001250,NaN,1,1
3,DECIPHER:1,HP:0001252,NaN,1,1
4,DECIPHER:1,HP:0001518,NaN,1,1


In [12]:
print("Known frequencies:",
      aggregated_frequencies["frequency"].notna().sum())

print("Unknown frequencies:",
      aggregated_frequencies["frequency"].isna().sum())

print("\nFrequency statistics:")
display(aggregated_frequencies["frequency"].describe())

Known frequencies: 215736
Unknown frequencies: 51843

Frequency statistics:


count    215736.000000
mean          0.530141
std           0.341363
min           0.000000
25%           0.170000
50%           0.545000
75%           0.895000
max           1.000000
Name: frequency, dtype: float64

In [12]:
aggregated_frequencies["frequency"].describe()

count    215736.000000
mean          0.530141
std           0.341363
min           0.000000
25%           0.170000
50%           0.545000
75%           0.895000
max           1.000000
Name: frequency, dtype: float64

In [13]:
print(
    "Pairs with frequency 0:",
    (aggregated_frequencies["frequency"] == 0).sum()
)

print(
    "Pairs with frequency 1:",
    (aggregated_frequencies["frequency"] == 1).sum()
)

Pairs with frequency 0: 3018
Pairs with frequency 1: 41210


In [14]:
disease_frequency_stats = (
    aggregated_frequencies
    .assign(
        is_zero=lambda df: df["frequency"] == 0,
        is_positive=lambda df: df["frequency"] > 0,
        is_unknown=lambda df: df["frequency"].isna()
    )
    .groupby("database_id")
    .agg(
        total_phenotypes=("hpo_id", "count"),
        known_positive=("is_positive", "sum"),
        zero_frequency=("is_zero", "sum"),
        unknown_frequency=("is_unknown", "sum")
    )
    .reset_index()
)

display(disease_frequency_stats.head(10))

,database_id,total_phenotypes,known_positive,zero_frequency,unknown_frequency
0,DECIPHER:1,5,0,0,5
1,DECIPHER:14,5,0,0,5
2,DECIPHER:15,6,0,0,6
3,DECIPHER:16,5,0,0,5
4,DECIPHER:17,5,0,0,5
5,DECIPHER:18,6,0,0,6
6,DECIPHER:19,4,0,0,4
7,DECIPHER:2,3,0,0,3
8,DECIPHER:20,5,0,0,5
9,DECIPHER:21,5,0,0,5


In [15]:
print("Number of diseases:", len(disease_frequency_stats))

print("\nTotal phenotypes per disease:")
display(disease_frequency_stats["total_phenotypes"].describe())

print("\nKnown positive phenotypes per disease:")
display(disease_frequency_stats["known_positive"].describe())

print("\nZero-frequency phenotypes per disease:")
display(disease_frequency_stats["zero_frequency"].describe())

print("\nUnknown-frequency phenotypes per disease:")
display(disease_frequency_stats["unknown_frequency"].describe())

Number of diseases: 12866

Total phenotypes per disease:


count    12866.000000
mean        20.797373
std         20.089846
min          1.000000
25%          6.000000
50%         15.000000
75%         28.000000
max        230.000000
Name: total_phenotypes, dtype: float64


Known positive phenotypes per disease:


count    12866.000000
mean        16.533344
std         19.452694
min          0.000000
25%          2.000000
50%         11.000000
75%         24.000000
max        226.000000
Name: known_positive, dtype: float64


Zero-frequency phenotypes per disease:


count    12866.000000
mean         0.234572
std          1.152906
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         49.000000
Name: zero_frequency, dtype: float64


Unknown-frequency phenotypes per disease:


count    12866.000000
mean         4.029457
std          7.813905
min          0.000000
25%          0.000000
50%          0.000000
75%          5.000000
max         80.000000
Name: unknown_frequency, dtype: float64

In [16]:
print(disease_frequency_stats[
    ["total_phenotypes", "known_positive",
     "zero_frequency", "unknown_frequency"]
].describe().to_string())

       total_phenotypes  known_positive  zero_frequency  unknown_frequency
count      12866.000000    12866.000000    12866.000000       12866.000000
mean          20.797373       16.533344        0.234572           4.029457
std           20.089846       19.452694        1.152906           7.813905
min            1.000000        0.000000        0.000000           0.000000
25%            6.000000        2.000000        0.000000           0.000000
50%           15.000000       11.000000        0.000000           0.000000
75%           28.000000       24.000000        0.000000           5.000000
max          230.000000      226.000000       49.000000          80.000000


In [17]:
diseases_without_known_frequency = disease_frequency_stats[
    disease_frequency_stats["known_positive"] == 0
]

print(
    "Diseases without known positive frequencies:",
    len(diseases_without_known_frequency)
)

print(
    "\nPercentage:",
    round(
        len(diseases_without_known_frequency)
        / len(disease_frequency_stats) * 100,
        2
    ),
    "%"
)

display(diseases_without_known_frequency.head(10))

Diseases without known positive frequencies: 2548

Percentage: 19.8 %


,database_id,total_phenotypes,known_positive,zero_frequency,unknown_frequency
0,DECIPHER:1,5,0,0,5
1,DECIPHER:14,5,0,0,5
2,DECIPHER:15,6,0,0,6
3,DECIPHER:16,5,0,0,5
4,DECIPHER:17,5,0,0,5
5,DECIPHER:18,6,0,0,6
6,DECIPHER:19,4,0,0,4
7,DECIPHER:2,3,0,0,3
8,DECIPHER:20,5,0,0,5
9,DECIPHER:21,5,0,0,5


## 4. Build disease-level frequency profiles

We now create a mapping of the form:

`disease_id -> {HPO_term: probability_or_None}`

This representation is used by the synthetic-patient generator. We then inspect representative profiles and summarize how many diseases have:
- at least one known positive phenotype frequency;
- all phenotype frequencies unknown;
- no known positive frequency, but at least one known zero frequency.


In [18]:
import importlib
import src.phenotype_frequency

importlib.reload(src.phenotype_frequency)

from src.phenotype_frequency import (
    build_disease_frequency_profiles
)

disease_frequency_profiles = build_disease_frequency_profiles(
    aggregated_frequencies
)

In [19]:
print("Number of diseases:", len(disease_frequency_profiles))

sample_disease = "DECIPHER:1"

print("\nDisease:", sample_disease)
print("Phenotype frequencies:")
print(disease_frequency_profiles[sample_disease])

print(
    "\nNumber of phenotypes:",
    len(disease_frequency_profiles[sample_disease])
)

Number of diseases: 12866

Disease: DECIPHER:1
Phenotype frequencies:
{'HP:0000252': None, 'HP:0001249': None, 'HP:0001250': None, 'HP:0001252': None, 'HP:0001518': None}

Number of phenotypes: 5


In [20]:
sample_mixed_disease = next(
    disease_id
    for disease_id, frequencies in disease_frequency_profiles.items()
    if any(value is not None for value in frequencies.values())
    and any(value is None for value in frequencies.values())
)

print("Disease:", sample_mixed_disease)
print("Phenotype frequencies:")
print(disease_frequency_profiles[sample_mixed_disease])

Disease: OMIM:100100
Phenotype frequencies:
{'HP:0000028': None, 'HP:0000072': None, 'HP:0000126': None, 'HP:0000217': 1.0, 'HP:0000767': None, 'HP:0000768': None, 'HP:0001374': None, 'HP:0001562': None, 'HP:0001643': None, 'HP:0001762': None, 'HP:0002023': None, 'HP:0004392': 0.4, 'HP:0005199': None, 'HP:0010957': 1.0, 'HP:0030211': 1.0}


In [21]:
expected_phenotypes_per_disease = (
    aggregated_frequencies
    .dropna(subset=["frequency"])
    .groupby("database_id")["frequency"]
    .sum()
)

mean_expected_phenotypes = expected_phenotypes_per_disease.mean()

print(
    "Diseases with known frequencies:",
    len(expected_phenotypes_per_disease)
)

print(
    "Mean expected selected phenotypes:",
    mean_expected_phenotypes
)

Diseases with known frequencies: 10338
Mean expected selected phenotypes: 11.06311101199003


In [22]:
# Diseases where every phenotype frequency is unknown
fully_unknown_diseases = disease_frequency_stats[
    (disease_frequency_stats["unknown_frequency"] ==
     disease_frequency_stats["total_phenotypes"])
]

# Diseases with known frequencies, but no positive frequency
no_positive_frequency_diseases = disease_frequency_stats[
    (disease_frequency_stats["known_positive"] == 0) &
    (disease_frequency_stats["unknown_frequency"] <
     disease_frequency_stats["total_phenotypes"])
]

print("Diseases with all frequencies unknown:",
      len(fully_unknown_diseases))

print("Diseases with known frequencies but no positive values:",
      len(no_positive_frequency_diseases))

Diseases with all frequencies unknown: 2528
Diseases with known frequencies but no positive values: 20


In [23]:
eligible_diseases = disease_frequency_stats[
    disease_frequency_stats["known_positive"] > 0
]["database_id"]

mean_expected_phenotypes = (
    aggregated_frequencies[
        aggregated_frequencies["database_id"].isin(eligible_diseases)
    ]
    .groupby("database_id")["frequency"]
    .sum()
    .mean()
)

print("Eligible diseases:", len(eligible_diseases))
print("Mean expected selected phenotypes:", mean_expected_phenotypes)

Eligible diseases: 10318
Mean expected selected phenotypes: 11.084555305480997


## 5. Test phenotype sampling

The sampling function has three possible outcomes:

- `frequency_weighted`: each phenotype with a known positive frequency is independently selected with that probability; zero-frequency terms are not selected.
- `uniform`: when all frequencies for a disease are unknown, up to `unknown_frequency_sample_size` terms are sampled uniformly.
- `ineligible`: no phenotype can be selected under the current policy (for example, a profile with no positive known frequencies and at least one known zero).

These small controlled profiles test the three branches before using real disease profiles.


In [24]:
from src.synthetic_patients import sample_phenotypes_from_frequency_profile

# Test 1: Known positive frequencies
known_profile = {
    "HP:0000001": 1.0,
    "HP:0000002": 0.0,
    "HP:0000003": 0.5,
    "HP:0000004": 0.8
}

result_1 = sample_phenotypes_from_frequency_profile(
    known_profile,
    random_seed=42
)

print("Test 1 - Known frequencies")
print(result_1)


# Test 2: All frequencies unknown
unknown_profile = {
    "HP:0000001": None,
    "HP:0000002": None,
    "HP:0000003": None,
    "HP:0000004": None,
    "HP:0000005": None
}

result_2 = sample_phenotypes_from_frequency_profile(
    unknown_profile,
    unknown_frequency_sample_size=3,
    random_seed=42
)

print("\nTest 2 - Unknown frequencies")
print(result_2)


# Test 3: No positive known frequencies
zero_profile = {
    "HP:0000001": 0.0,
    "HP:0000002": 0.0,
    "HP:0000003": None
}

result_3 = sample_phenotypes_from_frequency_profile(
    zero_profile,
    random_seed=42
)

print("\nTest 3 - No positive frequencies")
print(result_3)

Test 1 - Known frequencies
{'phenotype_terms': {'HP:0000003', 'HP:0000001', 'HP:0000004'}, 'sampling_strategy': 'frequency_weighted'}

Test 2 - Unknown frequencies
{'phenotype_terms': {'HP:0000003', 'HP:0000001', 'HP:0000005'}, 'sampling_strategy': 'uniform'}

Test 3 - No positive frequencies
{'phenotype_terms': set(), 'sampling_strategy': 'ineligible'}


## 6. Reload and import the current generator implementation

The generator is implemented in `src/synthetic_patients.py`, not redefined in this notebook.

After editing a module in `src/`, reload it before importing the functions again. This is especially important after a kernel restart or when testing a recent code change.


In [25]:
import src.synthetic_patients
importlib.reload(src.synthetic_patients)

from src.synthetic_patients import (
    sample_phenotypes_from_frequency_profile,
    generate_synthetic_patient_for_disease,
    generate_synthetic_patients
)

print("Synthetic-patient functions imported from src.synthetic_patients.")


Synthetic-patient functions imported from src.synthetic_patients.


## 7. Generate and validate one synthetic patient

We first choose a disease with at least one known positive phenotype frequency. The same random seed should produce the same phenotype profile in repeated runs.


In [26]:
known_disease = next(
    disease_id
    for disease_id, profile in disease_frequency_profiles.items()
    if any(
        frequency is not None and frequency > 0
        for frequency in profile.values()
    )
)

patient = generate_synthetic_patient_for_disease(
    disease_id=known_disease,
    disease_frequency_profiles=disease_frequency_profiles,
    random_seed=42
)

print("Patient:", patient)
print("Number of phenotypes:", len(patient["phenotype_terms"]))

Patient: {'true_disease_id': 'DECIPHER:81', 'phenotype_terms': {'HP:0000369', 'HP:0000377', 'HP:0000343', 'HP:0000325', 'HP:0000293', 'HP:0000276', 'HP:0000431', 'HP:0000256', 'HP:0001167', 'HP:0009890', 'HP:0002167', 'HP:0001367', 'HP:0000347', 'HP:0000077', 'HP:0000407', 'HP:0001999', 'HP:0001382', 'HP:0000448', 'HP:0001249', 'HP:0001363', 'HP:0000494'}, 'sampling_strategy': 'frequency_weighted', 'is_valid': True}
Number of phenotypes: 21


In [27]:
patient_1 = generate_synthetic_patient_for_disease(
    disease_id=known_disease,
    disease_frequency_profiles=disease_frequency_profiles,
    random_seed=42
)

patient_2 = generate_synthetic_patient_for_disease(
    disease_id=known_disease,
    disease_frequency_profiles=disease_frequency_profiles,
    random_seed=42
)

print("Same phenotype terms:",
      patient_1["phenotype_terms"] == patient_2["phenotype_terms"])

print("Same sampling strategy:",
      patient_1["sampling_strategy"] == patient_2["sampling_strategy"])

Same phenotype terms: True
Same sampling strategy: True


## 8. Test batch generation on a small sample

Before generating patients for every eligible disease, we test the batch generator on three disease profiles.

We verify:
- the expected number of patients is generated per disease;
- patient identifiers are unique;
- the generator is reproducible when the same seed is used;
- the validity of generated phenotype profiles.


In [28]:
sample_profiles = dict(
    list(disease_frequency_profiles.items())[:3]
)

sample_patients = generate_synthetic_patients(
    disease_frequency_profiles=sample_profiles,
    n_per_disease=100,
    random_seed=42
)

print("Total patients:", len(sample_patients))
print("First patient:", sample_patients[0])

Total patients: 300
First patient: {'true_disease_id': 'DECIPHER:1', 'phenotype_terms': {'HP:0000252', 'HP:0001250', 'HP:0001249', 'HP:0001252', 'HP:0001518'}, 'sampling_strategy': 'uniform', 'is_valid': True, 'patient_id': 'DECIPHER:1_patient_1'}


In [29]:
from collections import Counter

patients_per_disease = Counter(
    patient["true_disease_id"]
    for patient in sample_patients
)

patient_ids = [
    patient["patient_id"]
    for patient in sample_patients
]

print("Patients per disease:", patients_per_disease)
print("All diseases have 100 patients:",
      all(count == 100 for count in patients_per_disease.values()))
print("Unique patient IDs:", len(set(patient_ids)))
print("Total patient IDs:", len(patient_ids))

Patients per disease: Counter({'DECIPHER:1': 100, 'DECIPHER:14': 100, 'DECIPHER:15': 100})
All diseases have 100 patients: True
Unique patient IDs: 300
Total patient IDs: 300


In [30]:
valid_patients = [
    patient for patient in sample_patients
    if patient["is_valid"]
]

invalid_patients = [
    patient for patient in sample_patients
    if not patient["is_valid"]
]

print("Valid patients:", len(valid_patients))
print("Invalid patients:", len(invalid_patients))
print(
    "Invalid percentage:",
    len(invalid_patients) / len(sample_patients) * 100
)

Valid patients: 300
Invalid patients: 0
Invalid percentage: 0.0


In [31]:
patients_run_1 = generate_synthetic_patients(
    sample_profiles,
    n_per_disease=100,
    random_seed=42
)

patients_run_2 = generate_synthetic_patients(
    sample_profiles,
    n_per_disease=100,
    random_seed=42
)

same_results = all(
    p1["patient_id"] == p2["patient_id"]
    and p1["phenotype_terms"] == p2["phenotype_terms"]
    and p1["sampling_strategy"] == p2["sampling_strategy"]
    for p1, p2 in zip(patients_run_1, patients_run_2)
)

print("Reproducible generation:", same_results)

Reproducible generation: True


## 9. Classify diseases by sampling strategy

A disease is eligible if it has at least one known positive frequency, or if all its phenotype frequencies are unknown (in which case the uniform fallback is used).

Diseases with some known frequencies but no positive values are currently classified as ineligible. They are retained in the summary so that exclusions remain transparent.


In [32]:
from collections import Counter

disease_strategies = Counter(
    "frequency_weighted"
    if any(
        frequency is not None and frequency > 0
        for frequency in profile.values()
    )
    else "uniform"
    if profile and all(
        frequency is None
        for frequency in profile.values()
    )
    else "ineligible"
    for profile in disease_frequency_profiles.values()
)

print("Disease sampling strategies:")
for strategy, count in disease_strategies.items():
    print(f"{strategy}: {count}")

print("Total diseases:", sum(disease_strategies.values()))

Disease sampling strategies:
uniform: 2528
frequency_weighted: 10318
ineligible: 20
Total diseases: 12866


In [33]:
eligible_disease_profiles = {
    disease_id: profile
    for disease_id, profile in disease_frequency_profiles.items()
    if (
        any(freq is not None and freq > 0 for freq in profile.values())
        or (
            profile
            and all(freq is None for freq in profile.values())
        )
    )
}

print("Eligible diseases:", len(eligible_disease_profiles))

Eligible diseases: 12846


In [34]:
print("Project root:", PROJECT_ROOT)
print("Project root exists:", PROJECT_ROOT.exists())

print("Output directory:", PROJECT_ROOT / "data" / "processed")

print("Number of eligible diseases:", len(eligible_disease_profiles))

print(
    "Expected number of patients:",
    len(eligible_disease_profiles) * 100
)

assert PROJECT_ROOT.exists()
assert len(eligible_disease_profiles) == 12846
assert len(eligible_disease_profiles) * 100 == 1_284_600

print("\nAll checks passed!")

Project root: D:\uni_ 3\ProiectCercetare\hpo-similarity-synthetic-vs-clinical
Project root exists: True
Output directory: D:\uni_ 3\ProiectCercetare\hpo-similarity-synthetic-vs-clinical\data\processed
Number of eligible diseases: 12846
Expected number of patients: 1284600

All checks passed!


## 10. Validate the retry behavior on the eligible disease set

For frequency-weighted sampling, independent Bernoulli selection can occasionally produce an empty phenotype profile. The patient-level generator therefore retries sampling up to `max_attempts` times and returns a valid patient as soon as at least one phenotype is selected.

This produces profiles distributed according to the Bernoulli sampling process conditional on selecting at least one phenotype. If all attempts fail, the patient is marked invalid rather than entering an unbounded loop.

The batch test below generates 10 patients per eligible disease. This is an intermediate validation run, not the final experimental dataset.


In [34]:
test_patients = generate_synthetic_patients(
    disease_frequency_profiles=eligible_disease_profiles,
    n_per_disease=10,
    random_seed=42
)

print("Total patients:", len(test_patients))

valid_patients = sum(
    patient["is_valid"] for patient in test_patients
)

invalid_patients = len(test_patients) - valid_patients

print("Valid patients:", valid_patients)
print("Invalid patients:", invalid_patients)
print("Invalid percentage:", invalid_patients / len(test_patients) * 100)

Total patients: 128460
Valid patients: 128449
Invalid patients: 11
Invalid percentage: 0.008562976802117392


In [35]:
from collections import Counter

invalid_by_strategy = Counter(
    patient["sampling_strategy"]
    for patient in test_patients
    if not patient["is_valid"]
)

print("Invalid patients by strategy:")
for strategy, count in invalid_by_strategy.items():
    print(f"{strategy}: {count}")

Invalid patients by strategy:
frequency_weighted: 11


## 11. Focused regression test for one disease

Finally, we test the patient-level function directly. The returned `phenotype_terms` should be a set, not a string, and `sampling_strategy` should be one of the documented strategy labels.


In [36]:
test_disease_id = known_disease

patient_1 = generate_synthetic_patient_for_disease(
    disease_id=test_disease_id,
    disease_frequency_profiles=disease_frequency_profiles,
    random_seed=42
)

patient_2 = generate_synthetic_patient_for_disease(
    disease_id=test_disease_id,
    disease_frequency_profiles=disease_frequency_profiles,
    random_seed=42
)

print("Valid:", patient_1["is_valid"])
print("Phenotype container type:", type(patient_1["phenotype_terms"]))
print("Number of phenotypes:", len(patient_1["phenotype_terms"]))
print("Sampling strategy:", patient_1["sampling_strategy"])
print(
    "Reproducible:",
    patient_1["phenotype_terms"] == patient_2["phenotype_terms"]
)

assert isinstance(patient_1["phenotype_terms"], set)
assert patient_1["is_valid"] is True
assert len(patient_1["phenotype_terms"]) > 0
assert patient_1["sampling_strategy"] in {
    "frequency_weighted", "uniform"
}
assert patient_1["phenotype_terms"] == patient_2["phenotype_terms"]

print("Focused patient-generation test passed.")


Valid: True
Phenotype container type: <class 'set'>
Number of phenotypes: 21
Sampling strategy: frequency_weighted
Reproducible: True
Focused patient-generation test passed.


## 12. Interpretation and next steps

After running the notebook from top to bottom, record:
- the number of eligible and ineligible diseases;
- the number of synthetic patients generated;
- the percentage of invalid patients after the retry mechanism;
- the distribution of phenotype counts per patient;
- the distribution of sampling strategies.

The next stage is to define and implement the perturbation scenarios (noise, imprecision, phenotype dropout, and combined perturbations), then evaluate disease ranking using the selected similarity methods.

**Important:** the current notebook validates the baseline frequency-aware generation process. It does not yet implement or evaluate all perturbation scenarios.



## 13. Investigate invalid synthetic patients

Although the retry mechanism substantially reduced the number of invalid patients, a small number of patients still have empty phenotype profiles after the maximum number of sampling attempts.

We investigate these cases to determine whether they are associated with diseases having particularly low phenotype frequencies or very small phenotype profiles.

The objective is to understand the source of these invalid profiles before deciding whether these diseases should be excluded or handled differently.


In [38]:
# Identify invalid patients.
invalid_patients = [
    patient
    for patient in test_patients
    if not patient["is_valid"]
]

print("Number of invalid patients:", len(invalid_patients))

# Identify the diseases associated with invalid patients.
invalid_disease_ids = sorted({
    patient["true_disease_id"]
    for patient in invalid_patients
})

print("Number of affected diseases:", len(invalid_disease_ids))
print("\nAffected disease IDs:")

for disease_id in invalid_disease_ids:
    print(disease_id)

Number of invalid patients: 11
Number of affected diseases: 8

Affected disease IDs:
OMIM:106280
OMIM:157600
OMIM:227650
OMIM:269150
OMIM:613793
OMIM:617898
OMIM:618031
OMIM:620442



## 14. Analyze disease profiles associated with invalid patients

We examine the disease frequency profiles associated with invalid synthetic patients.

For each affected disease, we calculate:
- the total number of annotated phenotypes;
- the number of phenotypes with known positive frequencies;
- the number of phenotypes with zero frequencies;
- the mean and maximum positive phenotype frequency;
- the theoretical probability of generating an empty phenotype profile in a single sampling attempt.

For independent Bernoulli sampling, the probability of selecting no phenotypes is:

\[
P(\text{empty}) = \prod_{i=1}^{n}(1-p_i)
\]

where \(p_i\) is the frequency-based selection probability of phenotype \(i\).

This analysis helps determine whether repeated sampling failures are explained by the frequency distributions of the affected diseases.


In [39]:
# Analyze frequency profiles of diseases with invalid patients.

invalid_disease_analysis = []

for disease_id in invalid_disease_ids:

    profile = disease_frequency_profiles[disease_id]

    positive_frequencies = [
        freq for freq in profile.values()
        if freq is not None and freq > 0
    ]

    zero_frequencies = [
        freq for freq in profile.values()
        if freq == 0
    ]

    unknown_frequencies = [
        freq for freq in profile.values()
        if freq is None
    ]

    # Theoretical probability of selecting no phenotype.
    probability_empty = 1.0

    for freq in positive_frequencies:
        probability_empty *= (1 - freq)

    invalid_count = sum(
        patient["true_disease_id"] == disease_id
        and not patient["is_valid"]
        for patient in test_patients
    )

    invalid_disease_analysis.append({
        "disease_id": disease_id,
        "total_phenotypes": len(profile),
        "positive_frequencies": len(positive_frequencies),
        "zero_frequencies": len(zero_frequencies),
        "unknown_frequencies": len(unknown_frequencies),
        "mean_positive_frequency": (
            sum(positive_frequencies) / len(positive_frequencies)
            if positive_frequencies else None
        ),
        "max_positive_frequency": (
            max(positive_frequencies)
            if positive_frequencies else None
        ),
        "probability_empty": probability_empty,
        "invalid_patients": invalid_count
    })

invalid_disease_df = pd.DataFrame(invalid_disease_analysis)

invalid_disease_df.sort_values(
    by="probability_empty",
    ascending=False
).reset_index(drop=True)

,disease_id,total_phenotypes,positive_frequencies,zero_frequencies,unknown_frequencies,mean_positive_frequency,max_positive_frequency,probability_empty,invalid_patients
0,OMIM:620442,1,1,0,0,0.010081,0.010081,0.989919,4
1,OMIM:106280,2,1,0,1,0.025000,0.025000,0.975000,1
2,OMIM:269150,61,1,0,60,0.025000,0.025000,0.975000,1
3,OMIM:227650,34,1,0,33,0.025000,0.025000,0.975000,1
4,OMIM:617898,6,1,0,5,0.025000,0.025000,0.975000,1
5,OMIM:613793,1,1,0,0,0.025000,0.025000,0.975000,1
6,OMIM:618031,2,1,0,1,0.025000,0.025000,0.975000,1
7,OMIM:157600,3,2,0,1,0.025000,0.025000,0.950625,1



## 15. Investigate diseases with mixed known and unknown frequencies

The previous analysis revealed that some diseases have many annotated phenotypes but only one or a few phenotypes with known positive frequencies.

Under the current sampling policy, phenotypes with unknown frequencies are ignored whenever at least one known positive frequency is available.

We therefore quantify how frequently this situation occurs across the eligible disease profiles.

This analysis will help determine whether the current sampling policy systematically produces incomplete synthetic patient profiles.


In [40]:
# Identify diseases with both known positive and unknown frequencies.

mixed_frequency_diseases = []

for disease_id, profile in eligible_disease_profiles.items():

    positive_count = sum(
        freq is not None and freq > 0
        for freq in profile.values()
    )

    unknown_count = sum(
        freq is None
        for freq in profile.values()
    )

    if positive_count > 0 and unknown_count > 0:
        mixed_frequency_diseases.append({
            "disease_id": disease_id,
            "positive_frequencies": positive_count,
            "unknown_frequencies": unknown_count,
            "total_phenotypes": len(profile),
            "unknown_percentage": (
                unknown_count / len(profile) * 100
            )
        })

mixed_frequency_df = pd.DataFrame(mixed_frequency_diseases)

print("Diseases with mixed frequencies:", len(mixed_frequency_df))
print(
    "Percentage of eligible diseases:",
    len(mixed_frequency_df) / len(eligible_disease_profiles) * 100
)

mixed_frequency_df.sort_values(
    by="unknown_percentage",
    ascending=False
).head(10)

Diseases with mixed frequencies: 3750
Percentage of eligible diseases: 29.19196637085474


,disease_id,positive_frequencies,unknown_frequencies,total_phenotypes,unknown_percentage
597,OMIM:218040,1,71,72,98.611111
722,OMIM:234100,1,65,66,98.484848
137,OMIM:129900,1,61,62,98.387097
1027,OMIM:269150,1,60,61,98.360656
1262,OMIM:301041,1,48,49,97.959184
3094,OMIM:616897,1,47,48,97.916667
1982,OMIM:610883,1,47,48,97.916667
1373,OMIM:314580,1,47,48,97.916667
5,OMIM:101400,1,46,47,97.872340
527,OMIM:208400,1,45,46,97.826087



## 16. Distribution of known positive phenotype frequencies

The previous analysis identified 3,750 diseases with mixed known and unknown phenotype frequencies.

We now examine how many phenotypes have known positive frequencies in these disease profiles.

This analysis helps quantify how much information is available for frequency-weighted sampling and whether the current sampling policy systematically relies on a very limited subset of the annotated phenotypes.


In [41]:
# Count known positive frequencies per disease.

positive_frequency_counts = mixed_frequency_df[
    "positive_frequencies"
]

frequency_count_distribution = (
    positive_frequency_counts
    .value_counts()
    .sort_index()
    .reset_index()
)

frequency_count_distribution.columns = [
    "positive_phenotype_count",
    "number_of_diseases"
]

frequency_count_distribution["percentage"] = (
    frequency_count_distribution["number_of_diseases"]
    / len(mixed_frequency_df) * 100
)

print("Distribution of positive phenotype counts:")
display(frequency_count_distribution)

print("\nSummary statistics:")
print(positive_frequency_counts.describe())

Distribution of positive phenotype counts:


,positive_phenotype_count,number_of_diseases,percentage
0,1,337,8.986667
1,2,237,6.320000
2,3,182,4.853333
3,4,160,4.266667
4,5,192,5.120000
...,...,...,...
95,119,2,0.053333
96,131,1,0.026667
97,142,1,0.026667
98,161,1,0.026667



Summary statistics:
count    3750.000000
mean       14.566933
std        15.048820
min         1.000000
25%         5.000000
50%        10.000000
75%        19.750000
max       182.000000
Name: positive_frequencies, dtype: float64



## 17. Group diseases by the number of known positive frequencies

To better understand the availability of frequency information, we group diseases with mixed frequency profiles according to the number of phenotypes with known positive frequencies.

The groups allow us to distinguish diseases with very limited frequency information from those with a larger number of usable phenotype frequencies.

This analysis provides a clearer overview of the potential impact of incomplete frequency annotations on synthetic patient generation.


In [42]:
# Group diseases according to the number of known positive frequencies.

bins = [0, 1, 5, 10, 20, float("inf")]
labels = [
    "1 phenotype",
    "2-5 phenotypes",
    "6-10 phenotypes",
    "11-20 phenotypes",
    "More than 20 phenotypes"
]

mixed_frequency_df["positive_frequency_group"] = pd.cut(
    mixed_frequency_df["positive_frequencies"],
    bins=bins,
    labels=labels
)

frequency_groups = (
    mixed_frequency_df["positive_frequency_group"]
    .value_counts(sort=False)
    .rename("number_of_diseases")
    .reset_index()
)

frequency_groups["percentage"] = (
    frequency_groups["number_of_diseases"]
    / len(mixed_frequency_df) * 100
)

display(frequency_groups)

,positive_frequency_group,number_of_diseases,percentage
0,1 phenotype,337,8.986667
1,2-5 phenotypes,771,20.560000
2,6-10 phenotypes,785,20.933333
3,11-20 phenotypes,981,26.160000
4,More than 20 phenotypes,876,23.360000



## 18. Expected phenotype count under frequency-weighted sampling

For each disease with mixed frequency annotations, we calculate the expected number of selected phenotypes.

Under independent Bernoulli sampling, the expected number of selected phenotypes is:

\[
E[X] = \sum_{i=1}^{n} p_i
\]

where \(p_i\) represents the known positive frequency of phenotype \(i\).

We also calculate the theoretical probability of obtaining an empty phenotype profile:

\[
P(X=0) = \prod_{i=1}^{n}(1-p_i)
\]

These quantities help distinguish diseases with many annotated phenotypes from diseases for which the available frequency information supports only a small expected synthetic patient profile.


In [45]:
# Calculate expected phenotype counts and empty-profile probabilities.

expected_profile_stats = []

for disease_id, profile in eligible_disease_profiles.items():

    positive_frequencies = [
        freq for freq in profile.values()
        if freq is not None and freq > 0
    ]

    unknown_count = sum(
        freq is None for freq in profile.values()
    )

    if not positive_frequencies or unknown_count == 0:
        continue

    expected_count = sum(positive_frequencies)

    probability_empty = 1.0
    for freq in positive_frequencies:
        probability_empty *= (1 - freq)

    expected_profile_stats.append({
        "disease_id": disease_id,
        "positive_phenotypes": len(positive_frequencies),
        "unknown_phenotypes": unknown_count,
        "expected_phenotype_count": expected_count,
        "probability_empty": probability_empty
    })

expected_profile_df = pd.DataFrame(expected_profile_stats)

print("Diseases analyzed:", len(expected_profile_df))
print("\nExpected phenotype count:")
print(expected_profile_df["expected_phenotype_count"].describe().to_string())

print("\nProbability of an empty profile:")
print(expected_profile_df["probability_empty"].describe().to_string())

Diseases analyzed: 3750

Expected phenotype count:
count    3750.000000
mean        9.020327
std         9.655040
min         0.025000
25%         3.000000
50%         6.644253
75%        12.000000
max       162.647857

Probability of an empty profile:
count    3750.000000
mean        0.061287
std         0.197350
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         0.975000



## 19. Distribution of theoretical empty-profile probabilities

We examine the distribution of theoretical empty-profile probabilities across diseases with mixed frequency annotations.

Diseases are grouped according to the probability of generating an empty phenotype profile in a single Bernoulli sampling attempt.

This analysis identifies diseases for which the current frequency-weighted sampling strategy is likely to produce sparse or empty synthetic patient profiles.


In [47]:
# Group diseases by theoretical empty-profile probability.

thresholds = [-0.001, 0.10, 0.50, 0.90, 1.001]

labels = [
    "0%–10%",
    "10%–50%",
    "50%–90%",
    "90%–100%"
]

expected_profile_df["empty_probability_group"] = pd.cut(
    expected_profile_df["probability_empty"],
    bins=thresholds,
    labels=labels
)

empty_probability_distribution = (
    expected_profile_df["empty_probability_group"]
    .value_counts(sort=False)
    .rename("number_of_diseases")
    .reset_index()
)

empty_probability_distribution["percentage"] = (
    empty_probability_distribution["number_of_diseases"]
    / len(expected_profile_df) * 100
)

print(empty_probability_distribution.to_string(index=False))

empty_probability_group  number_of_diseases  percentage
                 0%–10%                3367   89.786667
                10%–50%                 147    3.920000
                50%–90%                 215    5.733333
               90%–100%                  21    0.560000



## 20. Compare theoretical and observed generation failures

We compare the theoretical probability of generating an empty phenotype profile with the observed number of invalid patients after applying the retry mechanism.

For each disease associated with invalid patients, we examine:
- its theoretical probability of generating an empty profile in one attempt;
- the probability of failing across all 100 attempts;
- the observed number of invalid patients.

Assuming independent sampling attempts, the probability of 100 consecutive failures is:

\[
P(\text{100 failures}) = P(\text{empty})^{100}
\]

This comparison helps assess whether the remaining invalid patients are consistent with the theoretical sampling process.


In [48]:
# Compare theoretical and observed failures.

invalid_comparison = expected_profile_df[
    expected_profile_df["disease_id"].isin(invalid_disease_ids)
].copy()

invalid_comparison["probability_100_failures"] = (
    invalid_comparison["probability_empty"] ** 100
)

observed_invalid_counts = (
    pd.Series(
        [p["true_disease_id"] for p in invalid_patients]
    )
    .value_counts()
    .rename("observed_invalid_patients")
)

invalid_comparison = invalid_comparison.merge(
    observed_invalid_counts,
    left_on="disease_id",
    right_index=True,
    how="left"
)

print(
    invalid_comparison[
        [
            "disease_id",
            "probability_empty",
            "probability_100_failures",
            "observed_invalid_patients"
        ]
    ].to_string(index=False)
)

 disease_id  probability_empty  probability_100_failures  observed_invalid_patients
OMIM:106280           0.975000                  0.079517                          1
OMIM:157600           0.950625                  0.006323                          1
OMIM:227650           0.975000                  0.079517                          1
OMIM:269150           0.975000                  0.079517                          1
OMIM:617898           0.975000                  0.079517                          1
OMIM:618031           0.975000                  0.079517                          1



## 21. Complete analysis of all invalid-patient cases

The previous comparison included only diseases with mixed frequency annotations. However, invalid patients may also originate from diseases whose phenotype frequencies are entirely known.

To ensure complete coverage, we calculate the theoretical empty-profile probability directly from the original disease frequency profiles for all diseases associated with invalid patients.

This allows us to compare all observed failures against the corresponding theoretical probabilities.


In [49]:
# Analyze all diseases associated with invalid patients.

complete_invalid_analysis = []

for disease_id in invalid_disease_ids:

    profile = disease_frequency_profiles[disease_id]

    positive_frequencies = [
        freq for freq in profile.values()
        if freq is not None and freq > 0
    ]

    probability_empty = 1.0

    for freq in positive_frequencies:
        probability_empty *= (1 - freq)

    observed_count = sum(
        patient["true_disease_id"] == disease_id
        and not patient["is_valid"]
        for patient in invalid_patients
    )

    complete_invalid_analysis.append({
        "disease_id": disease_id,
        "total_phenotypes": len(profile),
        "positive_frequencies": len(positive_frequencies),
        "probability_empty": probability_empty,
        "probability_100_failures": probability_empty ** 100,
        "observed_invalid_patients": observed_count
    })

complete_invalid_df = pd.DataFrame(complete_invalid_analysis)

print(complete_invalid_df.to_string(index=False))

 disease_id  total_phenotypes  positive_frequencies  probability_empty  probability_100_failures  observed_invalid_patients
OMIM:106280                 2                     1           0.975000                  0.079517                          1
OMIM:157600                 3                     2           0.950625                  0.006323                          1
OMIM:227650                34                     1           0.975000                  0.079517                          1
OMIM:269150                61                     1           0.975000                  0.079517                          1
OMIM:613793                 1                     1           0.975000                  0.079517                          1
OMIM:617898                 6                     1           0.975000                  0.079517                          1
OMIM:618031                 2                     1           0.975000                  0.079517                          1
OMIM:620


## 22. Conclusions on frequency-weighted sampling and retry behavior

The analysis of invalid synthetic patients revealed that all observed failures originated from the frequency-weighted sampling strategy.

These failures are consistent with the theoretical probability of repeatedly obtaining empty phenotype profiles. Diseases with only one or a few known positive phenotype frequencies can have a substantial probability of generating empty profiles, even after 100 independent sampling attempts.

The retry mechanism substantially reduced the invalid-patient rate from approximately 2.01% to 0.0086% in the preliminary experiment.

The current sampling strategy is therefore retained:
- Known positive phenotype frequencies are used as independent Bernoulli selection probabilities.
- Diseases with entirely unknown frequencies use uniform sampling.
- Diseases without any known positive frequencies are excluded from synthetic patient generation.
- Empty profiles are regenerated up to a maximum of 100 attempts.
- Patients that remain invalid are excluded from downstream similarity evaluation and reported separately.

Unknown phenotype frequencies are not interpreted as zero and are not assigned arbitrary probabilities.

This policy preserves the available frequency information without introducing unsupported assumptions.



## 23. Final synthetic patient generation

This section generates the final synthetic patient dataset using
the validated frequency-based sampling procedure.

The experiment uses:
- 100 synthetic patients per eligible disease;
- a maximum of 100 sampling attempts per patient;
- a fixed random seed to ensure reproducibility;
- batches of 100 diseases to limit memory consumption.

Generated patients are saved incrementally in JSON Lines format.
Invalid patients are retained in the output and reported separately.
They will be excluded from downstream similarity evaluation.


In [ ]:

import json
import random
from collections import Counter

# Final experiment parameters
N_PER_DISEASE = 100
MAX_ATTEMPTS = 100
UNKNOWN_FREQUENCY_SAMPLE_SIZE = 11
RANDOM_SEED = 42
BATCH_SIZE = 100

# Expected dataset size
expected_diseases = len(eligible_disease_profiles)
expected_patients = expected_diseases * N_PER_DISEASE

print("Eligible diseases:", expected_diseases)
print("Expected synthetic patients:", expected_patients)

assert expected_diseases == 12846
assert expected_patients == 1_284_600

# Create output directory
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "synthetic_patients.jsonl"

# Avoid accidentally overwriting an existing dataset
if OUTPUT_FILE.exists():
    raise FileExistsError(
        f"{OUTPUT_FILE} already exists. "
        "Choose another output filename or inspect the existing dataset."
    )

# Prepare reproducible batch seeds
seed_generator = random.Random(RANDOM_SEED)

disease_ids = sorted(eligible_disease_profiles)

# Statistics collected during generation
total_patients = 0
valid_patients = 0
invalid_patients = 0

strategy_counts = Counter()
phenotype_counts = Counter()

# Generate and save patients incrementally
with OUTPUT_FILE.open("w", encoding="utf-8") as output_file:

    for start in range(0, len(disease_ids), BATCH_SIZE):

        batch_ids = disease_ids[start:start + BATCH_SIZE]

        batch_profiles = {
            disease_id: eligible_disease_profiles[disease_id]
            for disease_id in batch_ids
        }

        batch_seed = seed_generator.randint(0, 2**32 - 1)

        batch_patients = generate_synthetic_patients(
            disease_frequency_profiles=batch_profiles,
            n_per_disease=N_PER_DISEASE,
            unknown_frequency_sample_size=UNKNOWN_FREQUENCY_SAMPLE_SIZE,
            random_seed=batch_seed,
            max_attempts=MAX_ATTEMPTS
        )

        # Convert phenotype sets to sorted lists for JSON serialization
        for patient in batch_patients:

            patient_record = {
                **patient,
                "phenotype_terms": sorted(patient["phenotype_terms"])
            }

            output_file.write(
                json.dumps(patient_record) + "\n"
            )

            total_patients += 1

            if patient["is_valid"]:
                valid_patients += 1
            else:
                invalid_patients += 1

            strategy_counts.update([patient["sampling_strategy"]])
            phenotype_counts.update([len(patient["phenotype_terms"])])

        output_file.flush()

        print(
            f"Processed diseases: "
            f"{min(start + BATCH_SIZE, len(disease_ids))}"
            f"/{len(disease_ids)}"
        )

# Final generation statistics

assert total_patients == expected_patients
assert valid_patients + invalid_patients == total_patients

print("Dataset size validation passed.")

print("\nGeneration completed.")
print("Total patients:", total_patients)
print("Valid patients:", valid_patients)
print("Invalid patients:", invalid_patients)

print(
    "Invalid percentage:",
    invalid_patients / total_patients * 100
)

print("\nSampling strategies:")
print(strategy_counts)

print("\nPhenotype count distribution:")
print(phenotype_counts)

print("\nDataset saved to:", OUTPUT_FILE)

assert sum(strategy_counts.values()) == total_patients